# **2. CLASSEMENT SELON DES THEMES**

In [ ]:
from dotenv import load_dotenv
from transformers import AutoTokenizer, XLMRobertaForSequenceClassification, AutoModelForSequenceClassification, pipeline
import os
import pandas as pd
from huggingface_hub import login
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from mistralai import Mistral
import time

from utils import add_prefix, run_damo_xlm, run_facebook_bart, run_xlm_roberta_xnli, run_mdeberta_xnli, compare_models, get_topic_keywords, classify

load_dotenv()

In [4]:
# Chargement des themes depuis le fichier texte
with open("data/themes.txt", "r", encoding="utf-8") as f:
    themes = [line.strip() for line in f if line.strip()]

print(themes)

['distance avec les autres villes', 'types de population', 'esthétique de la ville', 'histoire de la toponymie urbaine', 'élement géomorphologique', 'périphérie de ville', 'porte', 'voirie', 'pouvoir', 'palais ; citadelle et château', 'mur et murailles', 'matériaux', 'infrastructure de services', 'eau potable et fontaine', 'pont', 'port et bateau', 'habitation et maison', 'marché', 'richesse de la ville ou du prince', 'impôts et taxes', 'espaces agricoles', 'artisanat', 'mosquée', 'église', 'synagogue', 'cimetière et tombes', 'monuments et vestiges', 'pyramide', 'jardin', 'climat', 'animaux', 'plantes et arbres']


In [5]:
# Chargement du dataframe exporté en fin de première partie
df_chunks = pd.read_excel('data/df_chunks_part1.xlsx')

## **2.1. Zero-shot-classifaction**

Les quatre modèles que nous utilisons pour la classification zero-shot reposent tous sur des architectures de type Transformer, mais présentent des différences liées à leurs bases pré-entraînées et à leurs objectifs spécifiques.
1. Le modèle `DAMO-NLP-SG/zero-shot-classify-SSTuning-XLM-R` est basé sur XLM-RoBERTa et optimisé pour le **multilingue**, avec un entraînement spécial pour la classification zero-shot, ce qui le rend adapté à des textes dans plusieurs langues.
2. Le modèle `facebook/bart-large-mnli` est une version fine-tunée du modèle BART, entraîné sur la tâche NLI (Natural Language Inference) en anglais, souvent utilisé en zero-shot grâce à sa capacité à comprendre des relations entre phrases, mais son multilinguisme est plus limité.
3. Le modèle `joeddav/xlm-roberta-large-xnli` est basé sur XLM-RoBERTa large et fine-tuné sur le dataset XNLI, une tâche NLI multilingue, ce qui pourrait être un bon choix pour le zero-shot multilingue, notamment sur des langues comme l’arabe ou l’hébreu.
4. Enfin, `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` est une variante récente de DeBERTa, fine-tunée sur **MNLI et XNLI**, combinant des améliorations architecturales (comme un meilleur décodage du contexte) avec un apprentissage sur des données multilingues, ce qui le rend performant en classification zero-shot dans plusieurs langues.

In [3]:
# Téléchargement pour DAMO XLM-R (zero-shot multilingue)
print("Téléchargement : DAMO-NLP-SG/zero-shot-classify-SSTuning-XLM-R")
AutoTokenizer.from_pretrained("DAMO-NLP-SG/zero-shot-classify-SSTuning-XLM-R")
AutoModelForSequenceClassification.from_pretrained("DAMO-NLP-SG/zero-shot-classify-SSTuning-XLM-R")

# Téléchargement pour Facebook BART Large MNLI
print("Téléchargement : facebook/bart-large-mnli")
pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Téléchargement pour XLM-RoBERTa Large XNLI
print("Téléchargement : joeddav/xlm-roberta-large-xnli")
pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli")

# Téléchargement pour mDeBERTa v3 base XNLI
print("Téléchargement : MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")


Téléchargement : DAMO-NLP-SG/zero-shot-classify-SSTuning-XLM-R
Téléchargement : facebook/bart-large-mnli


Device set to use mps:0


Téléchargement : joeddav/xlm-roberta-large-xnli


Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


Téléchargement : MoritzLaurer/mDeBERTa-v3-base-mnli-xnli


Device set to use mps:0


In [4]:
# Configuration
HF_API_KEY = os.getenv('HF_API_KEY')
login(token=HF_API_KEY)

models = {
    "XLM-R (DAMO)": run_damo_xlm,
    "Facebook BART": run_facebook_bart,
    "XLM-R XNLI": run_xlm_roberta_xnli,
    "mDeBERTa XNLI": run_mdeberta_xnli,
}

In [6]:
# Exécution
text = df_chunks.loc[553, 'CHUNK_MOTS']
df_comparaison = compare_models(text, themes, models)
df_comparaison = df_comparaison.style.apply(lambda x: ['background-color: yellow' if val in x.nlargest(4).values else '' for val in x])
print(text)
df_comparaison

Running model: XLM-R (DAMO)
Running model: Facebook BART


Device set to use cpu


Running model: XLM-R XNLI


Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Running model: mDeBERTa XNLI


Device set to use cpu


testati sunt nobis redeunes, se numquam in plano fortius vidisse castrum: septem turres habet firmissimas testudinatas, insuper per girum duplici fossato et utroque murato cingitur habens antemurale.


,XLM-R (DAMO),Facebook BART,XLM-R XNLI,mDeBERTa XNLI
distance avec les autres villes,81.740000,29.210000,31.490000,28.120000
types de population,21.510000,30.110000,49.660000,61.910000
esthétique de la ville,32.730000,13.450000,85.140000,89.710000
histoire de la toponymie urbaine,50.740000,5.480000,53.060000,48.750000
élement géomorphologique,58.740000,11.660000,99.920000,99.910000
périphérie de ville,48.800000,3.060000,0.960000,94.570000
porte,49.300000,11.590000,3.670000,0.500000
voirie,69.310000,4.820000,66.810000,97.180000
pouvoir,35.040000,13.290000,99.760000,98.940000
palais ; citadelle et château,57.100000,2.030000,99.420000,82.060000


> **Traduction en français du passage latin**: « Les nôtres, quand ils s'en revinrent de cette expédition, nous affirmèrent que, jamais, en terrain plat, ils n'avaient vu une citadelle plus puissante; elle possède sept tours très fortes et couvertes en dôme; en outre, sur tout son pourtour, elle est ceinte d'un double fossé, et chacun de ses fossés est défendu pat un mur ; en plus, elle possède un avant-mur. »

Les thèmes ici devrait être :
* palais ; citadelle et château ("ils n'avaient vu une citadelle plus puissante")
* mur et murailles ("elle est ceinte d'un double fossé, et chacun de ses fossés est défendu pat un mur ; en plus, elle possède un avant-mur")

Seul le modèle XLM-R XNLI, dont le score des 3 premiers thèmes les plus élevé correspond au thèmes que nous avons choisi. Toutefois, lorsque l'on regarde de plus prêts, il attribue le thème animaux avec un score de 98% ; or à aucun moment il est question d'animaux  dans ce chunks-là. Cela rend ainsi compliqué le choix de la limite de confiance à attriber

Essayons un autre façon de faire !

## **2.2. BERTopic**

In [19]:
topic_list = [
    ["distance", "ville"],
    ["type", "population"],
    ["esthétique", "ville"],
    ["histoire", "ville","toponymie"],
    ["périphérie","ville"],
    ["porte","ville"],
    ["voirie","ville"],
    ["pouvoir","ville"],
    ["palais","citadelle","chateau"],
    ["mur","murailles"],
    ["matériaux","ville"],
    ["service","ville", "infrastructure"],
    ["eau"],
    ["fontaine"],
    ["pont"],
    ["habitation","maison"],
    ["port","bateau"],
    ["marché"],
    ["richesse","ville","prince"],
    ["impôts","taxes"],
    ["agriculture"],
    ["artisanat"],
    ["mosquée"],
    ["église"],
    ["synagogue"],
    ["cimetière","tombes"],
    ["monuments","antiquités"],
    ["jardin"],
    ["climat"],
    ["animaux"],
    ["plantes","arbres"],
    ["pyramide"],
    ["montagne", "fleuve","relief","mer","plaine"]
]


In [8]:
classification_bertopic_model= SentenceTransformer('sentence-transformers/LaBSE')


In [9]:
topic_model = BERTopic(embedding_model=classification_bertopic_model,
                       seed_topic_list=topic_list,
                       calculate_probabilities=True,
                       verbose=True)

In [10]:
topics, probs = topic_model.fit_transform(df_chunks['CHUNK_MOTS'])

2025-08-30 12:36:07,761 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 20/20 [00:25<00:00,  1.26s/it]
2025-08-30 12:36:33,280 - BERTopic - Embedding - Completed ✓
2025-08-30 12:36:33,293 - BERTopic - Guided - Find embeddings highly related to seeded topics.
Batches: 100%|██████████| 2/2 [00:04<00:00,  2.03s/it]
2025-08-30 12:36:37,610 - BERTopic - Guided - Completed ✓
2025-08-30 12:36:37,612 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-08-30 12:36:49,454 - BERTopic - Dimensionality - Completed ✓
2025-08-30 12:36:49,480 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-08-30 12:36:49,776 - BERTopic - Cluster - Completed ✓
2025-08-30 12:36:49,886 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-08-30 12:36:50,127 - BERTopic - Representation - Completed ✓


In [11]:
# Utilisation d'une fonction pour récupérer les mots-clés des seed topics correspondants

multi_results = get_topic_keywords(probs, topic_list, threshold=0.01)


In [12]:
chunks = df_chunks['CHUNK_MOTS'].tolist()

In [13]:
for result in multi_results:
    chunk_id = result['chunk_id']
    print(f"\nChunk {chunk_id}: '{chunks[chunk_id][:100]}...'")

    for topic_info in result['detected_topics']:
        keywords = topic_info['keywords']
        prob = topic_info['probability']
        print(f"  - {keywords} : {prob:.2%}")


Chunk 0: 'الغسطاط تعرف بباب اليون وهو الموضع المعروف بالقصر فلما افتح عمرو بن العاص باب اليون فى خلافة عمر بن ...'
  - ['distance', 'ville'] : 53.75%
  - ['type', 'population'] : 4.35%
  - ['esthétique', 'ville'] : 2.97%
  - ['histoire', 'ville', 'toponymie'] : 2.66%
  - ['périphérie', 'ville'] : 2.97%
  - ['porte', 'ville'] : 3.76%
  - ['voirie', 'ville'] : 4.69%
  - ['pouvoir', 'ville'] : 7.02%
  - ['palais', 'citadelle', 'chateau'] : 2.99%
  - ['mur', 'murailles'] : 6.31%
  - ['matériaux', 'ville'] : 2.48%
  - ['service', 'ville', 'infrastructure'] : 3.58%

Chunk 1: 'عمرو بن العاص مسجد جامعها ودار امارتها المعروفة بدار الرمل وجعل الاسواق محيطة بالمسجد لجامع فى الجان...'
  - ['distance', 'ville'] : 40.61%
  - ['type', 'population'] : 5.79%
  - ['esthétique', 'ville'] : 3.70%
  - ['histoire', 'ville', 'toponymie'] : 3.41%
  - ['périphérie', 'ville'] : 3.70%
  - ['porte', 'ville'] : 4.93%
  - ['voirie', 'ville'] : 6.03%
  - ['pouvoir', 'ville'] : 6.87%
  - ['palais', 'citadelle', 'cha

In [14]:
df_chunks.loc[536, 'CHUNK_MOTS']

' nota, quam cito aqua in decrescendo transit, ubicunque terra apparet, ibi statim rusticus aratrum figit et semen mittit. ln martio frumentum metunt. terra illa non parit aliud frumentum nisi tnticum et hordeum pulcherrimum omne genus leguminis a festo s.'

## **2.3. Mistral AI**

In [4]:
# Chargement de la clé API et paramètrage
MISTRAL_API_KEY = os.getenv('MISTRAL_API_KEY')
client = Mistral(api_key=MISTRAL_API_KEY)

In [7]:
# Appliquez la fonction classify avec un délai entre chaque appel
for index, row in df_chunks.iterrows():
    if index > 0 and index % 10 == 0:  # Ajoutez un délai toutes les 10 lignes, par exemple
        time.sleep(20)  # Attendez 60 secondes
    df_chunks.at[index, 'THEMES'] = classify(text_chunk=row['CHUNK_MOTS'], themes=themes, client=client)

Rate limit exceeded. Retrying in 1 seconds...
Rate limit exceeded. Retrying in 1 seconds...
Rate limit exceeded. Retrying in 1 seconds...
Rate limit exceeded. Retrying in 1 seconds...
Rate limit exceeded. Retrying in 1 seconds...
Rate limit exceeded. Retrying in 1 seconds...
Rate limit exceeded. Retrying in 1 seconds...
Rate limit exceeded. Retrying in 1 seconds...
Rate limit exceeded. Retrying in 1 seconds...


In [ ]:
# Tri des auteurs par ordre chronologique
# Nettoyer les noms d'auteurs (enlever les espaces superflus et convertir en majuscules)
df_chunks['AUTEUR'] = df_chunks['AUTEUR'].str.strip().str.upper()

ordre_chronologique = [
    "AL YAQUBI", "IBN RUSTAH", "AL-MASUDI", "IBN HAWQAL", "AL-MUQADDASI",
    "NAṢIR-I KHUSRAW", "ABU HAMID AL-GHARNATI", "AL-IDRISI", "HUGUES FALCAND",
    "GERARDUS BURCHARDUS", "BENJAMIN DE TUDELE", "GAUILLAUME DE TYR", "IBN JUBAYR",
    "THETMAR", "JACQUES DE VITRY", "OLIVIER", "VINCENT DE BEAUVAIS",
    "IBN SAID", "AL-HIMYARI"
]

# Ajuster les noms incorrects dans df_chunks si nécessaire
adjustments = {
    "NASIR-I KHOSRWO": "NAṢIR-I KHUSRAW", 
    "AL GHARNATI": "ABU HAMID AL-GHARNATI",
    "GUILLAUME DE TYR": "GAUILLAUME DE TYR",  # Ajustement de l'orthographe, si nécessaire
    "IBN  RUSTAH": "IBN RUSTAH"  # Suppression des espaces supplémentaires
}

# Appliquer les ajustements
df_chunks['AUTEUR'] = df_chunks['AUTEUR'].replace(adjustments)

# Fonction pour obtenir l'index dans ordre_chronologique, en gérant les valeurs manquantes
def get_order(auteur):
    try:
        return ordre_chronologique.index(auteur)
    except ValueError:
        # Si l'auteur n'est pas dans la liste, on lui donne un ordre arbitraire (par exemple, à la fin)
        return len(ordre_chronologique)

# Appliquer la fonction à la colonne 'AUTEUR'
df_chunks['ORDRE'] = df_chunks['AUTEUR'].apply(get_order)

# Trier le DataFrame selon l'ordre
df_chunks = df_chunks.sort_values('ORDRE').drop(columns=['ORDRE'])

In [27]:
# Sauvegarde du dataframe
df_chunks.to_excel("data/df_chunks_part2.xlsx", index=False)  # Sauvegarder df_chunks

## Premières analyses

In [31]:
df_chunks.head(5)                                   # Afficher df_chunks

,AUTEUR,LANGUE,VILLE,CHUNK_MOTS,NB_MOTS,CHUNK_ID,THEMES
0,AL YAQUBI,arabe,Le Caire,الغسطاط تعرف بباب اليون وهو الموضع المعروف بال...,50,1,"histoire de la toponymie urbaine, porte, pouvo..."
1,AL YAQUBI,arabe,Le Caire,عمرو بن العاص مسجد جامعها ودار امارتها المعروف...,36,2,"pouvoir, palais ; citadelle et château, marché..."
2,AL YAQUBI,arabe,Le Caire,واسكنه قوما وكتب الى عمر بن الخطاب بذلك فكتب ا...,50,3,"pouvoir, habitation et maison, richesse de la ..."
3,AL YAQUBI,arabe,Le Caire,لان لكل كورة مدينة مخصوصة بأمر من الامور من مد...,56,4,"histoire de la toponymie urbaine, artisanat, r..."
4,AL YAQUBI,arabe,Le Caire,ومدينة القيس وبها تعل الثياب القيسية والأكسية ...,50,5,"histoire de la toponymie urbaine, artisanat, e..."


In [32]:
x = 553
print(df_chunks.iloc[x, 3])
print(df_chunks.iloc[x, -1])

والحمرة بخطوط قبيحة مختلفة من كتب فقراء العامة إلا أن مع هذا كله على الجامع المذكور من الرونق, وحسن القبول, وانبساط النفس, ما لا تجده في جامع إشبيلية مع زخرفته, والبستان الذي في صحنه, وقد تأملت ما وجدت فيه من الارتياح والأنس دون منظر يوجب ذلك فعلمت أنه سر مودع
mosquée, jardin, richesse de la ville ou du prince


> **Traduction du passage :** Les nôtres, quand ils s'en revinrent de cette expédition, nous affirmèrent que, jamais, en terrain plat, ils n'avaient vu une citadelle plus puissante; elle possède sept tours très fortes et couvertes en dôme; en outre, sur tout son pourtour, elle est ceinte d'un double fossé, et chacun de ses fossés est défendu pat un mur ; en plus, elle possède un avant-mur.

In [33]:
# Nettoyage : transformation des chaînes de thèmes en listes
df_chunks['THEMES'] = df_chunks['THEMES'].str.split(',\s*')

# Pour chaque thème, créer une colonne avec True/False
for theme in themes:
    df_chunks[theme] = df_chunks['THEMES'].apply(lambda x: theme in x)

#Supprimer la colonne d’origine
df_chunks = df_chunks.drop(columns=['THEMES'])

# Affichage pour vérification
df_chunks.head(5)

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/var/folders/7m/x2ckvpzs0zg0lkkmq5_x00g00000gn/T/ipykernel_7217/2803523237.py:2: SyntaxWarning: invalid escape sequence '\s'
  df_chunks['THEMES'] = df_chunks['THEMES'].str.split(',\s*')


,AUTEUR,LANGUE,VILLE,CHUNK_MOTS,NB_MOTS,CHUNK_ID,distance avec les autres villes,types de population,esthétique de la ville,histoire de la toponymie urbaine,...,mosquée,église,synagogue,cimetière et tombes,monuments et vestiges,pyramide,jardin,climat,animaux,plantes et arbres
0,AL YAQUBI,arabe,Le Caire,الغسطاط تعرف بباب اليون وهو الموضع المعروف بال...,50,1,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,AL YAQUBI,arabe,Le Caire,عمرو بن العاص مسجد جامعها ودار امارتها المعروف...,36,2,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
2,AL YAQUBI,arabe,Le Caire,واسكنه قوما وكتب الى عمر بن الخطاب بذلك فكتب ا...,50,3,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,AL YAQUBI,arabe,Le Caire,لان لكل كورة مدينة مخصوصة بأمر من الامور من مد...,56,4,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
4,AL YAQUBI,arabe,Le Caire,ومدينة القيس وبها تعل الثياب القيسية والأكسية ...,50,5,False,False,False,True,...,False,False,False,False,False,False,False,False,False,True


In [34]:
df_chunks.shape

(609, 38)

In [35]:
cols_to_analyze = df_chunks.columns[6:]

# Compter les True et False pour chaque colonne
true_false_counts = {}
for col in cols_to_analyze:
  true_count = df_chunks[col].sum()
  false_count = (~df_chunks[col]).sum()
  true_false_counts[col] = {'True': true_count, 'False': false_count}

# Créer un DataFrame à partir des résultats
df_true_false = pd.DataFrame.from_dict(true_false_counts, orient='index')

# Afficher le DataFrame
df_true_false

,True,False
distance avec les autres villes,34,575
types de population,48,561
esthétique de la ville,54,555
histoire de la toponymie urbaine,60,549
élement géomorphologique,73,536
périphérie de ville,46,563
porte,42,567
voirie,20,589
pouvoir,167,442
palais ; citadelle et château,78,531


In [36]:
df_chunks.to_excel('data/df_chunks_part2_detailed.xlsx', index=False)